In [8]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_V2_S_Weights


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory():
    return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [9]:
csv_file = "results/EfficientNetV2s_metrics.csv"


if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])

## Loading dataset

In [10]:
# Pretrained EfficientNetV2-S weights
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1

# Use simple preprocessing only
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
#     num_workers=2,
#     pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
#     num_workers=2,
#     pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 68175
Val samples: 7575
Test samples: 25250


In [11]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [12]:
method_name = "batchnorm_finetune"

# Load pretrained model
model = models.efficientnet_v2_s(weights=weights)

# Replace classifier (same as before)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 101)

# Freeze EVERYTHING first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze BatchNorm layers
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        for param in module.parameters():
            param.requires_grad = True
        module.train()  # allow BN stats to update

## tune the classifier
for param in model.classifier.parameters():
    param.requires_grad = True
model = model.to(device)
# print(model)

In [13]:
criterion = nn.CrossEntropyLoss()

## optimizeer for the batch norm and the classifier
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [14]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # SAVE METRICS TO CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            method_name,
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params
        ])

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        model_path = f"models/{method_name}_EfficientNet_best.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, model_path)

        print("Best model saved!")

Trainable params: 283,253
Epoch 1/8
Train Loss: 1.5305, Train Acc: 0.6131
Val Loss:   0.8965, Val Acc:   0.7608
Time: 2051.64s | Peak Mem: 7139.04 MB
----------------------------------------
Best model saved!
Epoch 2/8
Train Loss: 0.9848, Train Acc: 0.7404
Val Loss:   0.7896, Val Acc:   0.7937
Time: 1907.34s | Peak Mem: 7134.94 MB
----------------------------------------
Best model saved!
Epoch 3/8
Train Loss: 0.8487, Train Acc: 0.7742
Val Loss:   0.7547, Val Acc:   0.8034
Time: 1906.79s | Peak Mem: 7134.94 MB
----------------------------------------
Best model saved!
Epoch 4/8
Train Loss: 0.7694, Train Acc: 0.7922
Val Loss:   0.7229, Val Acc:   0.8141
Time: 1904.13s | Peak Mem: 7134.94 MB
----------------------------------------
Best model saved!
Epoch 5/8
Train Loss: 0.7152, Train Acc: 0.8054
Val Loss:   0.7325, Val Acc:   0.8139
Time: 1991.73s | Peak Mem: 7134.94 MB
----------------------------------------
Epoch 6/8
Train Loss: 0.6714, Train Acc: 0.8155
Val Loss:   0.7082, Val Acc: 